In [1]:
import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
import torch.nn as nn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# List contents of 'MyDrive' to find your dataset folder
mydrive_path = '/content/drive/MyDrive'
if os.path.exists(mydrive_path):
    print("Files and folders in your Google Drive:")
    print(os.listdir(mydrive_path))
else:
    print("Drive not mounted correctly. Please run the mount cell above.")

Files and folders in your Google Drive:
['shah_fahad_quiz_reg_no_046.pdf', 'code.sh', 'Colab Notebooks', 'archive.zip', 'Dataset']


In [4]:
import zipfile

# Adjust this path based on what you found in Step 2 if it is different
correct_archive_path = '/content/drive/MyDrive/Colab Notebooks/dataset/archive.zip'
extract_destination = '/content/dataset'

if os.path.exists(correct_archive_path):
    with zipfile.ZipFile(correct_archive_path, 'r') as zip_ref:
        zip_ref.extractall(extract_destination)
    print(f"Extraction completed! Files extracted to: {extract_destination}")
    print("Extracted files:", os.listdir(extract_destination))
else:
    print(f"Could not find the zip file at: {correct_archive_path}. Please check the path and try again.")

Extraction completed! Files extracted to: /content/dataset
Extracted files: ['cifar10']


In [5]:
train_data_path='/content/dataset/cifar10/cifar10/train/'
test_data_path='/content/dataset/cifar10/cifar10/test/'

In [6]:
transform=transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.6,0.4],[0.5,0.6,0.4])

])

In [7]:
train_dataset=datasets.ImageFolder(root=train_data_path,transform=transform)
test_dataset=datasets.ImageFolder(root=test_data_path,transform=transform)


In [16]:
train_loader=DataLoader(train_dataset,
                        batch_size=16,
                        shuffle=True)
test_loader=DataLoader(test_dataset,
                       batch_size=16,
                       shuffle=False)


In [9]:
folder_label_names = train_dataset.classes
print("Detected labels from folder names:", folder_label_names)

Detected labels from folder names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [17]:
class AlexNetCifar10(nn.Module):
  def __init__(self,num_classes=10):
    super(AlexNetCifar10,self).__init__()
    self.features=nn.Sequential(
        #(3x32,32)->(64,32,32)
        nn.Conv2d(in_channels=3,out_channels=64,kernel_size=3,stride=1,padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2),#(64,16,16)

        nn.Conv2d(in_channels=64,out_channels=196,kernel_size=3,stride=1,padding=1),#(64,16,16)->(196,16,16)
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2),#(196,8,8)

        #(196,8,8)->(256,8,8)
        nn.Conv2d(in_channels=196,out_channels=256,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),

        #(256,8,8)->(256,8,8)
        nn.Conv2d(in_channels=256,out_channels=256,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2) #(256,4,4)
    )
    self.classifier=nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(256*4*4,4096),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(4096,4096),
        nn.ReLU(0.5),
        nn.Linear(4096,num_classes)
    )
  def forward(self,x):
    x=self.features(x)
    x=torch.flatten(x,1)
    x=self.classifier(x)

    return x

In [18]:
epochs=50
lr=0.001
training_loss=[]
testing_loss=[]
device='cuda' if torch.cuda.is_available() else 'cpu'
device


'cuda'

In [19]:
model=AlexNetCifar10().to(device)

In [20]:
import torch.optim as optim
optimizer=optim.AdamW(model.parameters(),lr=lr)
loss_fn=nn.CrossEntropyLoss()

In [22]:
from tqdm.notebook import tqdm

epoch_bar=tqdm(range(epochs),desc="training Progress")

for epoch in epoch_bar:
  model.train()
  running_loss=0.0
  batch_bar=tqdm(train_loader,desc=f"Epoch {epoch+1}/{epochs}",leave=False)

  for image,label in batch_bar:

    image,label=image.to(device),label.to(device)

    optimizer.zero_grad()
    pred=model(image)

    loss=loss_fn(pred,label)
    running_loss+=loss.item()

    loss.backward()
    optimizer.step()
    batch_bar.set_postfix(batch_loss=f"{loss.item():.4f}")

  epoch_loss=running_loss/len(train_loader)
  training_loss.append(epoch_loss)
  epoch_bar.set_postfix(epoch_loss=f"{epoch_loss:.4f}")

training Progress:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1/50:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 5/50:   0%|          | 0/3125 [00:00<?, ?it/s]

... (41 outputs left out)

Epoch 47/50:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 48/50:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/3125 [00:00<?, ?it/s]

In [23]:
torch.save(model.state_dict(),"AlexNet.pth")

In [24]:
model.eval()
test_loss=0.0
correct_predictions=0
total_sample=0
with torch.no_grad():

  for image,label in test_loader:
    image,label=image.to(device),label.to(device)

    pred=model(image)
    loss=loss_fn(pred,label)

    test_loss+=loss.item()*image.size(0)

    _,predicted=torch.max(pred,dim=1)

    total_sample+=image.size(0)
    correct_predictions+=(predicted==label).sum().item()
    testing_loss.append(loss.item())
  accuracy=(correct_predictions/total_sample)*100
  test_loss=test_loss/total_sample


print(f"Test Loss is :{test_loss :.4f}")
print(f"Test Accuracy is : {accuracy :.2f}%")

Test Loss is :0.6452
Test Accuracy is : 79.40%


In [33]:
def predict_new(model,image_path,label_names,device):
  image=Image.open(image_path).convert('RGB')
  transform=transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.6,0.4],[0.5,0.6,0.4])

])
  image_tensor=transform(image).to(device)
  image_tensor=image_tensor.unsqueeze(0)
  model.eval()
  with torch.no_grad():
    pred =model(image_tensor)
    pred_class=torch.argmax(pred).item()
  name=label_names[pred_class]

  return f"The item in the Picture is '{ name }' predicted by Model."


In [26]:
from google.colab import files
import os

# Prompt user to choose a local file
uploaded = files.upload()

# Get the name of the uploaded image
image_path = list(uploaded.keys())[0]
print(f"Successfully uploaded: {image_path}")


Saving download.jfif to download.jfif
Successfully uploaded: download.jfif


In [27]:
!dir

AlexNet.pth  dataset  download.jfif  drive  sample_data


In [31]:
image_path = "/content/download.jfif"
from PIL import Image

In [36]:
predict_new(model=model,image_path=image_path,label_names=folder_label_names,device=device)


"The item in the Picture is 'airplane' predicted by Model."